# Settings

## Import Packages

In [3]:
import pandas as pd
import numpy as np
import os
from scipy import stats
from scipy.stats import chi2_contingency, pointbiserialr

import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import math
import itertools
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
import requests
import time
from geopy.geocoders import Nominatim
import time
from shapely.wkt import dumps
from shapely.wkt import loads

pd.set_option('display.precision', 4)

## Self-defined functions

In [4]:
def load_geoDataFrame(filepath):
    """
    Load a GeoDataFrame from a CSV file with a geometry column in WKT format.

    Parameters:
        filepath (str): Path to the CSV file containing the saved GeoDataFrame.

    Returns:
        GeoDataFrame: A GeoDataFrame reconstructed from the CSV, with geometries parsed from WKT strings.
    
    Notes:
        - Assumes the geometry column is named 'geometry' and stored in WKT format.
        - The returned GeoDataFrame is assigned the CRS 'EPSG:4326' by default.
    """
    df = pd.read_csv(filepath)
    df['geometry'] = df['geometry'].apply(loads)
    return gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

In [5]:
def gen_valid_trip_influence_df(
    trip_df, start_time_col, end_time_col, valid_duration_hour,
    start_location_col, end_location_col, trip_id_col=None, location_ids=None
):
    """
    Generate time-location influence DataFrame from trip data with ±1 hour influence period.

    Parameters:
        trip_df (pd.DataFrame): Trip data containing timestamps and locations.
        start_time_col (str): Column name for trip start timestamp.
        end_time_col (str): Column name for trip end timestamp.
        valid_duration_hour (float): Maximum allowed trip duration (in hours).
        start_location_col (str): Column name for trip start location ID.
        end_location_col (str): Column name for trip end location ID.
        trip_id_col (str, optional): Column name for unique trip ID. If None, index will be used.
        location_ids (iterable, optional): If provided, only retain records where location_id is in this list.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]:
            - Cleaned original trip_df with standardized time columns and trip_id
            - Influence DataFrame with columns ['trip_id', 'trip_time', 'location_id']
    """
    trip_df = trip_df.copy()
    trip_df[start_time_col] = pd.to_datetime(trip_df[start_time_col])
    trip_df[end_time_col] = pd.to_datetime(trip_df[end_time_col])

    # Ensure correct time ordering
    trip_df['start_datetime'] = trip_df[[start_time_col, end_time_col]].min(axis=1)
    trip_df['end_datetime'] = trip_df[[start_time_col, end_time_col]].max(axis=1)

    # Filter by trip duration
    trip_df['duration'] = (trip_df['end_datetime'] - trip_df['start_datetime']).dt.total_seconds() / 3600
    trip_df = trip_df[trip_df['duration'].between(0, valid_duration_hour)]

    # Ensure trip_id exists
    if trip_id_col is None:
        trip_df['trip_id'] = trip_df.index
        trip_id_col = 'trip_id'

    # Round timestamps to the hour
    trip_df['start_rounded_hour'] = trip_df['start_datetime'].dt.floor('h')
    trip_df['end_rounded_hour'] = trip_df['end_datetime'].dt.floor('h')

    # Prepare event frames
    pickup_df = trip_df[[trip_id_col, 'start_rounded_hour', start_location_col]].copy()
    pickup_df.columns = ['trip_id', 'trip_time', 'location_id']

    dropoff_df = trip_df[[trip_id_col, 'end_rounded_hour', end_location_col]].copy()
    dropoff_df.columns = ['trip_id', 'trip_time', 'location_id']

    # Add influence hours
    pickup_influence_df = pickup_df.copy()
    pickup_influence_df['trip_time'] -= pd.Timedelta(hours=1)

    dropoff_influence_df = dropoff_df.copy()
    dropoff_influence_df['trip_time'] += pd.Timedelta(hours=1)

    # Combine and deduplicate
    trip_with_influence_df = pd.concat([pickup_df, pickup_influence_df, dropoff_df, dropoff_influence_df], ignore_index=True)

    if location_ids is not None:
        trip_with_influence_df = trip_with_influence_df[trip_with_influence_df['location_id'].isin(location_ids)]

    trip_with_influence_df = trip_with_influence_df.drop_duplicates()

    return trip_df, trip_with_influence_df

# Citi Bike Trip Histories
- Introduction: [Citi Bike Trip Histories](https://citibikenyc.com/system-data)
- Instructions:
  1. Download [files of Citi Bike trip data](https://s3.amazonaws.com/tripdata/index.html)
  2. Unzip all files in the `raw_data/` directory before running any parsing scripts.
- Date Range: 2024-04 to 2025-03
- The processed bike data has been saved to the [shared Google Drive folder](https://drive.google.com/drive/u/2/folders/1Yr7l0EcolHcL2TDrrq72ZUix2FNEc5X2)

## Create Merged Source Files

In [ ]:
root_folder = 'raw_data'
folders = [os.path.join(root_folder, name) for name in os.listdir(root_folder)
           if os.path.isdir(os.path.join(root_folder, name))]
folders

In [4]:
# # The code below was previously executed, and the resulting data has already been saved to CSV files.
# # To prevent unnecessary re-execution and preserve results, the code has been converted to comments.

# # Loop through each folder and read CSV files
# for folder in folders:
#     combined_df = pd.DataFrame()
#     for file in os.listdir(folder):
#         if file.endswith('.csv'):
#             file_path = os.path.join(folder, file)
#             print(f'Reading: {file_path}')
#             df = pd.read_csv(file_path)
#             print(len(df))
#             combined_df = pd.concat([combined_df, df], ignore_index=True)
#     combined_df.to_csv(f'{folder}.csv', index=False)
#     print(len(combined_df))

## Get Manhattan Bike Station ID 

In [6]:
# 1. Load Citi Bike station data (GBFS format)
stations = pd.read_json('https://gbfs.lyft.com/gbfs/2.3/bkn/en/station_information.json')['data']['stations']
station_df = pd.json_normalize(stations)

# 2. Convert to GeoDataFrame with EPSG:4326
gdf = gpd.GeoDataFrame(
    station_df,
    geometry=gpd.points_from_xy(station_df.lon, station_df.lat),
    crs='EPSG:4326'
)

# 3. Load Manhattan zone geometries and ensure uniqueness
manhattan_zones = load_geoDataFrame('../manhattan_grid/taxi_location_grid_mapping.csv')
manhattan_zones = manhattan_zones[['zone', 'geometry']].drop_duplicates()

# 4. Do spatial join to keep only stations within Manhattan zones
manhattan_station_df = gpd.sjoin(gdf, manhattan_zones, how='inner', predicate='within')
manhattan_bike_station_ids = manhattan_station_df['short_name'].unique()
len(manhattan_bike_station_ids)

676

In [ ]:
# manhattan_station_df.to_csv("bike_station_df.csv")

## Process Bike Data

In [16]:
final_result_df = pd.DataFrame()

folder = 'raw_data'
for file in os.listdir(folder):
    if file.endswith('.csv'):
        file_path = os.path.join(folder, file)
        print(f'Reading: {file_path}')
        source_bike_df = pd.read_csv(file_path, dtype={'start_station_id': str, 'end_station_id': str})
        print(len(source_bike_df))
        bike_df, bike_with_influence_df = gen_valid_trip_influence_df(trip_df=source_bike_df,
                                                                      start_time_col='started_at',
                                                                      end_time_col='ended_at',
                                                                      valid_duration_hour=3,
                                                                      start_location_col='start_station_id',
                                                                      end_location_col='end_station_id',
                                                                      trip_id_col='ride_id',
                                                                      location_ids=manhattan_bike_station_ids)
        result_df = bike_with_influence_df.groupby(['trip_time', 'location_id'])['trip_id'].count().reset_index(name='bike_trips')
        final_result_df = pd.concat([final_result_df, result_df], axis=0)

final_result_df.head()

Reading: raw_data/202404-citibike-tripdata.csv
3217063
Reading: raw_data/202412-citibike-tripdata.csv
2311171
Reading: raw_data/202406-citibike-tripdata.csv
4783576
Reading: raw_data/202410-citibike-tripdata.csv
5150054
Reading: raw_data/202503-citibike-tripdata.csv
3168271
Reading: raw_data/202408-citibike-tripdata.csv
4603575
Reading: raw_data/202501-citibike-tripdata.csv
2124475
Reading: raw_data/202405-citibike-tripdata.csv
4230360
Reading: raw_data/202411-citibike-tripdata.csv
3710134
Reading: raw_data/202407-citibike-tripdata.csv
4722896
Reading: raw_data/202409-citibike-tripdata.csv
4997898
Reading: raw_data/202502-citibike-tripdata.csv
2031257


,trip_time,location_id,bike_trips
0,2024-03-31 21:00:00,5914.03,2
1,2024-03-31 21:00:00,7580.01,3
2,2024-03-31 22:00:00,5024.10,2
3,2024-03-31 22:00:00,5137.11,1
4,2024-03-31 22:00:00,5216.04,1


In [ ]:
# Re-aggregate passenger counts by hour and location
final_result_df = final_result_df.groupby(['trip_time', 'location_id'])['bike_trips'].sum().reset_index()

# Filter data within the desired date range: 2024/04 - 2025/03
start_date = pd.to_datetime('2024-04-01')
end_date = pd.to_datetime('2025-04-01')
final_result_df = final_result_df[(final_result_df['trip_time'] >= start_date) & (final_result_df['trip_time'] < end_date)]

In [23]:
result_with_station_info = final_result_df.merge(manhattan_station_df[['short_name', 'lat', 'lon', 'geometry', 'name']], left_on='location_id', right_on='short_name')
result_with_station_info = result_with_station_info.drop(columns=['short_name'])

# Save the final aggregated data to CSV
result_with_station_info.to_csv('citi_bike_20240401_20250331.csv')
result_with_station_info.shape

(5042609, 7)